<div align="center">
  <a href="https://pageindex.ai/developer">
    <img src="https://github.com/user-attachments/assets/bae02956-6c4e-4a0b-adea-257b0be4aaa1" width="100%" alt="PageIndex — Vectorless, Reasoning-based RAG">
  </a>
  <p>
    <a href="https://pageindex.ai/developer">🌐 Website</a> &nbsp; · &nbsp;
    <a href="https://developer.pageindex.ai/">☁️ Cloud</a> &nbsp; · &nbsp;
    <a href="https://docs.pageindex.ai/">📖 Docs</a> &nbsp; · &nbsp;
    <a href="https://github.com/VectifyAI/PageIndex">⭐ GitHub</a> &nbsp; · &nbsp;
    <a href="https://pageindex.ai/contact">✉️ Contact</a>
  </p>
</div>

---

# PageIndex Metadata Search

**Find and count documents in a large library by their tags.**

Once a folder declares metadata fields and its documents carry tags, the chat agent can search the folder by them: filter documents by a field's value or range, sort them, and count or group them. This notebook demonstrates it on a library too large to read through.

The demo collection holds 120 10-K and 10-Q filings, about 11,500 pages, from six chipmakers: NVIDIA, AMD, Intel, Broadcom, Qualcomm, and Micron, five fiscal years each. Each filing keeps the name EDGAR gives it, its accession number, such as `0001045810-24-000264.pdf`, and is tagged with `company`, `form_type`, `fiscal_year`, and `fiscal_quarter`. Each company's fiscal year ends in a different month, so the tags say what a filing date alone does not.

1. **Read the fields** — See a folder's metadata fields with `get_metadata_schema()`.
2. **Ask by tag** — Filter and count filings by their tags in chat.
3. **Measure** — Inspect token usage and estimate the model cost.

**Before you start:** Have your OpenAI API key ready. This demo queries an existing, tagged collection on PageIndex Cloud with a read-only key, so no upload is needed. To declare fields and tag your own documents, see [PageIndex Document Metadata](https://github.com/VectifyAI/PageIndex/blob/main/cookbook/pageindex-metadata.ipynb).

## 01 · Set up

Install the latest PageIndex SDK.

In [ ]:
!pip install -U pageindex

### Configure your client

Set your API keys and choose your chat `MODEL`. The client queries the demo collection on PageIndex Cloud, and your chosen model answers with PageIndex Cloud's retrieval tools.

In [1]:
import os
from pageindex import PageIndexClient

os.environ["PAGEINDEX_API_KEY"] = "pageindex_demo_readonly_key"
os.environ["OPENAI_API_KEY"] = "Your OpenAI API key here"

MODEL = "gpt-5.6-luna"  # gpt-5.6-sol / gpt-5.6-terra / gpt-5.6-luna

client = PageIndexClient(
    index="cloud",
    chat=MODEL,
)

## 02 · Read the fields

`get_metadata_schema()` returns the fields that apply to a folder's documents, each with its type and description. The agent reads the same fields before it searches by them.

In [2]:
FOLDER_ID = "cmvoxrjpy002l0dnql7m69wh9"  # semiconductor-sec-filings

client.get_metadata_schema(FOLDER_ID)["schema"]

{'fields': {'company': {'type': 'string',
   'description': 'Filing company: NVIDIA, AMD, Intel, Broadcom, Qualcomm, or Micron'},
  'form_type': {'type': 'string',
   'description': 'SEC form: 10-K (annual report) or 10-Q (quarterly report)'},
  'fiscal_year': {'type': 'number',
   'description': "Fiscal year the company reports the filing under; each company's fiscal year ends in a different month (NVIDIA's FY2025 ended Jan 26, 2025)"},
  'fiscal_quarter': {'type': 'number',
   'description': 'Fiscal quarter 1-3 of a 10-Q; a 10-K has none'}}}

## 03 · Ask by tag

Ask for the third-quarter 10-Qs from fiscal 2024 onward. The agent reads the folder's fields with `get_metadata_schema`, then filters by `form_type`, `fiscal_quarter`, and a `fiscal_year` range with `search_documents(mode="metadata")`. The output streams the answer alongside retrieval tool calls.

In [3]:
messages = [{"role": "user", "content": "List the third-quarter 10-Qs from fiscal 2024 onward, with each company and fiscal year."}]

response = None
for event in client.chat(messages, folder_id=FOLDER_ID, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]


[Tool call] browse_documents({"folder_id":"cmvoxrjpy002l0dnql7m69wh9","recursive":true,"sort":"relevance","query":"third quarter 10-Q fiscal 2024","limit":50})



[Tool call] get_metadata_schema({})



[Tool call] search_documents({"mode":"metadata","folder_id":"cmvoxrjpy002l0dnql7m69wh9","recursive":true,"metadata":{"metadata_filter":{"form_type":"10-Q","fiscal_quarter":3,"fiscal_year":{"$gte":2024}},"sort":{"fiscal_year":1,"company":1},"aggregates":{"n":{"$count":{}}}},"limit":50})


Third

-quarter

...

 for

 fiscal

202

6

.

Tags also count. Ask how many 10-Ks and 10-Qs each company has: the agent groups the filings by `company` and `form_type` with `search_documents(mode="metadata")` and counts each group, without opening a filing.


In [4]:
messages = [{"role": "user", "content": "How many 10-K and 10-Q filings does each company have?"}]

response = None
for event in client.chat(messages, folder_id=FOLDER_ID, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]


[Tool call] browse_documents({"folder_id":"cmvoxrjpy002l0dnql7m69wh9","recursive":true,"sort":"time","limit":50})



[Tool call] get_metadata_schema({"folder_id":"cmvoxrjpy002l0dnql7m69wh9"})



[Tool call] search_documents({"mode":"metadata","folder_id":"cmvoxrjpy002l0dnql7m69wh9","recursive":true,"metadata":{"group_by":["company","form_type"],"aggregates":{"filings":{"$count":{}}},"sort":{"company":1,"form_type":1}},"limit":50})


|

 Company

...

 quarterly

10

-Q

s

.

## 04 · Review usage & estimated cost (optional)

Inspect the token usage for the last answer and estimate its model cost using the rate table below. Reading the fields does not require a model call. This estimate does not include PageIndex Cloud charges.


In [5]:
import json

# Standard USD per 1M tokens: (input, cached input, output).
# Checked 2026-09-17: https://developers.openai.com/api/docs/models/{model}
PRICES = {
    "gpt-5.6-sol": (4.0, 0.4, 20.0),
    "gpt-5.6-terra": (2.0, 0.2, 12.0),
    "gpt-5.6-luna": (0.2, 0.02, 1.2),
}
input_rate, cached_rate, output_rate = PRICES[MODEL]

usage = response.get("usage") if response else None
print("\n\nUsage:", json.dumps(usage, ensure_ascii=False, indent=2))
if response and response.get("status") != "completed":
    print("Response did not complete:", response.get("error") or response.get("incomplete_details"))

if usage:
    details = usage.get("input_tokens_details") or {}
    cached = details.get("cached_tokens", 0) or 0
    written = details.get("cache_write_tokens", 0) or 0
    regular = usage["input_tokens"] - cached - written
    cost = (regular * input_rate + cached * cached_rate + written * input_rate * 1.25
            + usage["output_tokens"] * output_rate) / 1_000_000
    print(f"Estimated model cost at base rates: ${cost:.6f} USD")
    if usage["input_tokens"] > 272_000:
        print("Usage is aggregated across model calls; the estimate excludes possible long-context surcharges.")
else:
    print("No usage returned; unable to estimate cost.")



Usage: {
  "input_tokens": 52163,
  "input_tokens_details": {
    "cached_tokens": 41482,
    "cache_write_tokens": 10669
  },
  "output_tokens": 394,
  "output_tokens_details": {
    "reasoning_tokens": 102
  },
  "total_tokens": 52557,
  "request_usage_entries": [
    {
      "input_tokens": 5462,
      "input_tokens_details": {
        "cached_tokens": 5459,
        "cache_write_tokens": 0
      },
      "output_tokens": 105,
      "output_tokens_details": {
        "reasoning_tokens": 57
      },
      "total_tokens": 5567
    },
    {
      "input_tokens": 15031,
      "input_tokens_details": {
        "cached_tokens": 5459,
        "cache_write_tokens": 9569
      },
      "output_tokens": 67,
      "output_tokens_details": {
        "reasoning_tokens": 31
      },
      "total_tokens": 15098
    },
    {
      "input_tokens": 15539,
      "input_tokens_details": {
        "cached_tokens": 15028,
        "cache_write_tokens": 508
      },
      "output_tokens": 96,
      "outpu

---

<div align="center">
  <br>
  <p>
    <a href="https://docs.pageindex.ai/"><b>Upload your documents</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://docs.pageindex.ai/sdk/agents"><b>Integrate with your agent</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://github.com/VectifyAI/PageIndex"><b>Star PageIndex on GitHub</b></a>
  </p>
  <br>
  <a href="https://pageindex.ai">
    <img src="https://github.com/user-attachments/assets/e1c677f2-b590-4ffc-859d-e8cc7d794bdf" width="144" height="40" alt="PageIndex">
  </a>
</div>
